# 02 - LDMH Unit Tests
**Addresses the Section VI claims and the module's validity (supporting comments 3 and 5).**

Checks, each printing PASS / FAIL (nothing is hidden):
1. model sizes match the paper (86,945 / 116,138 parameters);
2. the **clamp dead-zone** exists, and the diversity-penalty gradient vanishes for tied clamped betas (the paper's claim);
3. the **sigmoid is not exactly gradient-free in theory but IS in float32** beyond a finite pre-activation (the paper's 'never exactly zero' needs a caveat);
4. the old NLL's `clamp(min=eps)` created a zero-gradient region of its own - fixed in v2;
5. the **softmax temperature T=1.5 is really applied** (the submitted code never applied it);
6. **synthetic recovery**: can LDMH recover a known spatial partition of *different noise shapes* when it only sees the observable image? (The submitted test fed it the residual itself and scored 67.4%, below its own 85% threshold - never reported.)

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C.banner()
import torch.nn.functional as F
from models.ldmh import (LocalDistributionMixtureHead, mixture_gen_gaussian_nll, beta_diversity_penalty, load_balance_penalty_entropy)
from models.restoration_net import BaselineNet, DistributionMixtureRestorationNet
def check(name, ok, extra=""): print(("PASS " if ok else "FAIL ") + name, extra)
RESULTS = {}

MODE=pilot  epochs=30  seeds=[0, 1]  device=cpu  out=d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot


In [ ]:
n = lambda m: sum(p.numel() for p in m.parameters())
check("baseline parameter count == 86,945", n(BaselineNet()) == 86945, n(BaselineNet()))
check("full parameter count == 116,138", n(DistributionMixtureRestorationNet()) == 116138, n(DistributionMixtureRestorationNet()))

PASS baseline parameter count == 86,945 86945
PASS full parameter count == 116,138 116138


In [ ]:
# 2. clamp: tied betas beyond the ceiling -> zero gradient from the diversity penalty (paper Section VI-A)
l = torch.tensor([5.0, 5.2], requires_grad=True)
b = torch.exp(l).clamp(0.3, 2.5); beta_diversity_penalty(b).backward()
check("clamp: tied betas are bitwise identical", bool(b[0] == b[1]), b.tolist())
check("clamp: diversity-penalty gradient is exactly zero", bool((l.grad == 0).all()), l.grad.tolist())
l2 = torch.tensor([5.0, 5.2], requires_grad=True)
b2 = 0.3 + 2.2 * torch.sigmoid(l2); beta_diversity_penalty(b2).backward()
check("sigmoid: same configuration still receives gradient", bool((l2.grad != 0).all()), l2.grad.tolist())

PASS clamp: tied betas are bitwise identical [2.5, 2.5]
PASS clamp: diversity-penalty gradient is exactly zero [0.0, 0.0]
PASS sigmoid: same configuration still receives gradient [0.0146256722509861, -0.01200353167951107]


In [ ]:
# 3. float32 saturation of the sigmoid: where does the gradient become EXACTLY zero?
sat = None
for z in np.arange(5, 40, 0.5):
    t = torch.tensor([float(z)], requires_grad=True); (0.3 + 2.2 * torch.sigmoid(t)).sum().backward()
    if t.grad.item() == 0.0: sat = float(z); break
print("float32: d beta / d l becomes exactly 0 at pre-activation l >=", sat)
bound = lambda l: 0.3 + 2.2 / (1 + np.exp(-l))
print("beta at that point:", bound(sat), "-> the paper's 'never exactly zero' holds mathematically but NOT in float32; say 'practically vanishing'.")
RESULTS["sigmoid_float32_saturation_l"] = sat

float32: d beta / d l becomes exactly 0 at pre-activation l >= 17.0
beta at that point: 2.4999999089213736 -> the paper's 'never exactly zero' holds mathematically but NOT in float32; say 'practically vanishing'.


In [ ]:
# 4. the old NLL's clamp(min=eps) on r/scale: conceptually the same defect, but how large is the dead zone in practice?
def nll_old(res, w, beta, scale, eps=1e-6):
    K = beta.shape[0]; r = res.abs()
    b_, s_ = beta.view(1, K, 1, 1), scale.view(1, K, 1, 1)
    lc = torch.log(b_) - torch.log(2 * s_) - torch.lgamma(1.0 / b_) - (r / s_).clamp(min=eps).pow(b_)
    return -torch.logsumexp(torch.log(w + eps) + lc, 1).mean()
def grad_beta(fn, r0):
    res = torch.full((1, 1, 8, 8), r0); w = torch.full((1, 3, 8, 8), 1 / 3)
    be = torch.tensor([0.5, 1.0, 2.0], requires_grad=True); sc = torch.tensor([0.05] * 3, requires_grad=True)
    fn(res, w, be, sc).backward(); return be.grad.clone(), sc.grad.clone()
# isolate the |r/s|^beta term's gradient: difference between a tiny and a typical residual
for nm, fn in [("old (clamp)", nll_old), ("v2 (additive eps)", mixture_gen_gaussian_nll)]:
    g_tiny, _ = grad_beta(fn, 1e-9); g_zero, _ = grad_beta(fn, 0.0)
    print(f"{nm:18s} d NLL/d beta at |r|=1e-9: {g_tiny.tolist()}   |r|=0: {g_zero.tolist()}")
frac = float((np.abs(np.random.RandomState(0).normal(0, 0.085, 10 ** 6)) < 1e-6 * 0.05).mean())
print(f"fraction of N(0,0.085) residuals inside the old dead zone (|r| < eps*scale): {frac:.2e}")
check("old-NLL dead zone is negligible in practice (<1e-5 of pixels): v2 change is a cleanliness fix, NOT a source of gains", frac < 1e-5)

old (clamp)        d NLL/d beta at |r|=1e-9: [-0.7042274475097656, -0.16088926792144775, -0.003916993737220764]   |r|=0: [-0.7042274475097656, -0.16088926792144775, -0.003916993737220764]
v2 (additive eps)  d NLL/d beta at |r|=1e-9: [-0.7088098526000977, -0.16107071936130524, -0.003919616341590881]   |r|=0: [-0.7088073492050171, -0.16107060015201569, -0.0039196014404296875]
fraction of N(0,0.085) residuals inside the old dead zone (|r| < eps*scale): 0.00e+00
PASS old-NLL dead zone is negligible in practice (<1e-5 of pixels): v2 change is a cleanliness fix, NOT a source of gains 


In [ ]:
# 5. temperature is applied in v2
torch.manual_seed(0)
x = torch.randn(1, 1, 16, 16)
def ent(T):
    torch.manual_seed(1); m = LocalDistributionMixtureHead(temperature=T)
    w = m(x)[0]; return float(-(w * (w + 1e-9).log()).sum(1).mean())
e1, e15 = ent(1.0), ent(1.5)
check("T=1.5 gives flatter mixture weights than T=1.0 (temperature is applied)", e15 > e1, f"entropy {e1:.4f} -> {e15:.4f}")

PASS T=1.5 gives flatter mixture weights than T=1.0 (temperature is applied) entropy 1.0645 -> 1.0829


## 6. Synthetic recovery of a known partition
Truth: blob regions where residuals are heavy-tailed (beta 0.6) vs near-Gaussian (beta 1.9). The head sees only the **observable** noisy image (texture amplitude differs by region), not the residual. Repeated over 5 seeds. Thresholds fixed in advance: pixel agreement > 0.90 and each recovered beta within 25% of truth.

In [ ]:
from scipy import ndimage, stats
def make_field(seed, S=96):
    rng = np.random.RandomState(seed)
    mask = ndimage.gaussian_filter(rng.randn(S, S), 8) > 0
    resid = np.where(mask, stats.gennorm.rvs(0.6, scale=0.03, size=(S, S), random_state=rng), stats.gennorm.rvs(1.9, scale=0.05, size=(S, S), random_state=rng))
    tex = rng.randn(S, S) * np.where(mask, 0.25, 0.05)                           # observable structure differs by region
    return mask, resid.astype(np.float32), (0.5 + tex).astype(np.float32)
steps = 40 if C.MODE == "smoke" else 400
rows = []
for seed in range(5):
    mask, resid, img = make_field(seed)
    torch.manual_seed(seed)
    m = LocalDistributionMixtureHead(base_ch=16, n_components=2, init_scale=0.04)
    opt = torch.optim.Adam(m.parameters(), lr=5e-3)
    xi, rr = torch.from_numpy(img)[None, None], torch.from_numpy(resid)[None, None]
    for _ in range(steps):
        w, be, sc, _ = m(xi); loss = mixture_gen_gaussian_nll(rr, w, be, sc); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): w, be, sc, _ = m(xi)
    heavy = int(be.argmin()); pred = (w[0, heavy] > 0.5).numpy()
    agree = max((pred == mask).mean(), (pred != mask).mean())                  # label-switch invariant
    # context: a NON-learned detector of the same observable cue (local std of the input, Otsu threshold)
    from skimage.filters import threshold_otsu
    ls = np.sqrt(np.maximum(ndimage.uniform_filter(img ** 2, 5) - ndimage.uniform_filter(img, 5) ** 2, 0)); base_pred = ls > threshold_otsu(ls)
    base_agree = max((base_pred == mask).mean(), (base_pred != mask).mean())
    rows.append(dict(seed=seed, agreement=agree, local_std_baseline=base_agree, beta_lo=float(be.min()), beta_hi=float(be.max()), scale_lo=float(sc[be.argmin()]), scale_hi=float(sc[be.argmax()])))
rec = pd.DataFrame(rows); print(rec.round(3).to_string(index=False))
print("truth: beta 0.6 (scale 0.03) vs 1.9 (scale 0.05)")
ok_a = bool((rec.agreement > 0.90).all()); ok_o = bool(rec.agreement.mean() > 0.85)
ok_b = bool(((rec.beta_lo - 0.6).abs() / 0.6 < 0.25).all() and ((rec.beta_hi - 1.9).abs() / 1.9 < 0.25).all())
check("strict criterion (pre-registered): agreement > 0.90 in ALL seeds", ok_a, f"mean={rec.agreement.mean():.3f}")
check("original notebook-04 criterion: mean agreement > 0.85 (the submitted run scored 0.674)", ok_o)
check("prototype betas recovered within 25% of truth in all seeds", ok_b)
print(f"context: non-learned local-std baseline reaches {rec.local_std_baseline.mean():.3f}; LDMH {rec.agreement.mean():.3f}. "
      f"If LDMH is not clearly above this baseline, its spatial routing adds nothing beyond a simple local-variance cue.")
RESULTS.update(recovery_agreement_mean=float(rec.agreement.mean()), local_std_baseline_mean=float(rec.local_std_baseline.mean()),
               recovery_strict_pass=ok_a, recovery_original_criterion_pass=ok_o, beta_recovery_pass=ok_b)
if not (ok_a and ok_b): print("-> Report this honestly. Do NOT claim that LDMH 'learns local distribution structure' beyond what this shows.")
rec.to_csv(os.path.join(C.TABLES, "ldmh_synthetic_recovery.csv"), index=False)
json.dump(RESULTS, open(os.path.join(C.TABLES, "ldmh_unit_tests.json"), "w"), indent=1, default=float)

 seed  agreement  local_std_baseline  beta_lo  beta_hi  scale_lo  scale_hi
    0      0.878               0.970    0.492    2.245     0.027     0.054
    1      0.708               0.957    0.500    2.300     0.028     0.053
    2      0.844               0.970    0.492    2.119     0.027     0.051
    3      0.841               0.963    0.490    2.211     0.028     0.054
    4      0.715               0.970    0.496    2.229     0.028     0.054
truth: beta 0.6 (scale 0.03) vs 1.9 (scale 0.05)
FAIL strict criterion (pre-registered): agreement > 0.90 in ALL seeds mean=0.797
FAIL original notebook-04 criterion: mean agreement > 0.85 (the submitted run scored 0.674) 
PASS prototype betas recovered within 25% of truth in all seeds 
context: non-learned local-std baseline reaches 0.966; LDMH 0.797. If LDMH is not clearly above this baseline, its spatial routing adds nothing beyond a simple local-variance cue.
-> Report this honestly. Do NOT claim that LDMH 'learns local distribution structu